In [84]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
</style>
"""))

**<font size="6" color="red">ch06_RNN기반의 Seq2Seq(스마트번역기)</font>**
- Google Neural Machine Translation(GNMT)
- RNN기반의 Seq2Seq방식
- 인코더입력/디코더입력(모범답안) - 디코더 출력(답안) ; 인코더와 디코더가 연결된 구조

# 1. 패키지 import 및 하이퍼 파라미터


In [3]:
import numpy as np
import pandas as pd
from time import time

from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

# 하이퍼 파라미터
MY_HIDDEN = 128
MY_EPOCH = 500

# 2. 학습데이터

In [4]:
raw = pd.read_csv('data/translate.csv', header=None)
eng_kor=raw.values.tolist()
print(eng_kor[:3])
print('학습할 영-한 데이터 갯수:', len(eng_kor))

[['cold', '감기'], ['come', '오다'], ['cook', '요리']]
학습할 영-한 데이터 갯수: 110


In [5]:
e_alpha = [ c for c in 'SEPabcdefghijklmnopqrstuvwxyz']
korean = ''.join([ data[1] for data in eng_kor])
k_ch= list(set([ch for ch in korean]))
k_ch.sort()
k_alpha= k_ch

In [10]:
alpha = e_alpha + k_alpha
print('영어와 한글 알파벳 : ',alpha)
alpha_total_size= len(alpha)
print('전체 알파벳 갯수(원핫인코딩 사이즈):', alpha_total_size)

영어와 한글 알파벳 :  ['S', 'E', 'P', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '가', '각', '간', '감', '개', '거', '것', '게', '계', '고', '관', '광', '구', '굴', '규', '그', '금', '기', '깊', '나', '날', '남', '내', '넓', '녀', '노', '놀', '농', '높', '뉴', '늦', '다', '단', '도', '동', '들', '람', '랑', '래', '램', '류', '름', '릎', '리', '많', '망', '매', '머', '먼', '멍', '메', '명', '모', '목', '무', '물', '미', '바', '반', '방', '번', '복', '부', '분', '붕', '비', '뿌', '사', '상', '색', '생', '서', '선', '소', '손', '수', '쉽', '스', '시', '식', '실', '싸', '아', '약', '얇', '어', '언', '얼', '여', '연', '오', '옥', '왼', '요', '용', '우', '운', '움', '위', '유', '은', '을', '음', '의', '이', '익', '인', '읽', '입', '자', '작', '장', '적', '제', '좋', '주', '지', '짜', '쪽', '찾', '책', '출', '칙', '크', '키', '탈', '택', '통', '파', '팔', '편', '피', '핑', '한', '합', '해', '행', '험', '회', '획', '휴', '흐']
전체 알파벳 갯수(원핫인코딩 사이즈): 171


# 3. 문자당 num을 갖는 dict
- 전예제 : {'the':1, 'a':2,...} / {1:'the',2:'a',...}
- {'S':0, 'E':1,...}

In [15]:
# char_to_num={}
# for i, ch in enumerate(alpha):
#     char_to_num[ch] = i
char_to_num={ch: i for i, ch in enumerate(alpha)}
print(char_to_num)

{'S': 0, 'E': 1, 'P': 2, 'a': 3, 'b': 4, 'c': 5, 'd': 6, 'e': 7, 'f': 8, 'g': 9, 'h': 10, 'i': 11, 'j': 12, 'k': 13, 'l': 14, 'm': 15, 'n': 16, 'o': 17, 'p': 18, 'q': 19, 'r': 20, 's': 21, 't': 22, 'u': 23, 'v': 24, 'w': 25, 'x': 26, 'y': 27, 'z': 28, '가': 29, '각': 30, '간': 31, '감': 32, '개': 33, '거': 34, '것': 35, '게': 36, '계': 37, '고': 38, '관': 39, '광': 40, '구': 41, '굴': 42, '규': 43, '그': 44, '금': 45, '기': 46, '깊': 47, '나': 48, '날': 49, '남': 50, '내': 51, '넓': 52, '녀': 53, '노': 54, '놀': 55, '농': 56, '높': 57, '뉴': 58, '늦': 59, '다': 60, '단': 61, '도': 62, '동': 63, '들': 64, '람': 65, '랑': 66, '래': 67, '램': 68, '류': 69, '름': 70, '릎': 71, '리': 72, '많': 73, '망': 74, '매': 75, '머': 76, '먼': 77, '멍': 78, '메': 79, '명': 80, '모': 81, '목': 82, '무': 83, '물': 84, '미': 85, '바': 86, '반': 87, '방': 88, '번': 89, '복': 90, '부': 91, '분': 92, '붕': 93, '비': 94, '뿌': 95, '사': 96, '상': 97, '색': 98, '생': 99, '서': 100, '선': 101, '소': 102, '손': 103, '수': 104, '쉽': 105, '스': 106, '시': 107, '식': 108, '실': 109, '싸': 110,

In [8]:
# 문자-> 숫자 / 숫자-> 문자
print('문자->숫자:', char_to_num.get('c'))
print('숫자->문자:', alpha[5])

문자->숫자: 5
숫자->문자: c


In [24]:
data = eng_kor[0]
print(data)
print('인코더 입력(원핫인코딩전):', [char_to_num.get(ch) for ch in data[0]])
print('디코더 입력(원핫인코딩전):', [char_to_num.get(ch) for ch in 'S' + data[1]] )
print('디코더 출력:',[char_to_num.get(ch) for ch in data[1]+ 'E'])

['cold', '감기']
인코더 입력(원핫인코딩전): [5, 17, 14, 6]
디코더 입력(원핫인코딩전): [0, 32, 46]
디코더 출력: [32, 46, 1]


In [ ]:
# 원핫인코딩 방법1
pd.get_dummies([5, 17, 14, 6])

In [ ]:
# 원핫인코딩 방법2
to_categorical([5, 17, 14, 6], num_classes=alpha_total_size)

In [ ]:
#원핫인코딩 방법3 : np.eye(n) - n행 n열 단위행렬
np.eye(alpha_total_size)[[5, 17, 14, 6]]

# 4. 인코더입력, 디코더입력, 디코더출력
- 인코더입력과 디코더입력(원핫인코딩), 디코더출력(원핫인코딩X-loss를 sparse categoricalcrossentropy)


In [53]:
def encoding(eng_kor=eng_kor):
    '인코더입력데이터(110*4*171), 디코더입력데이터(110*3*171), 디코더출력데이터를 return'  # 3개 축이 다 동일해야함
       # 영어단어 110 , 문자4개, 원핫인코더 171
    enc_in=[] #인코더입력(cold 원핫인코딩)
    dec_in=[] #디코더입력(S 감기 원핫인코딩)
    dec_out=[] #디코더입력(감기E 라벨인코딩)
    for data in eng_kor:
        #인코더 입력(영어->숫자->원핫인코딩)
        eng=[char_to_num.get(ch) for ch in data[0]]
        eng_one=to_categorical(eng, num_classes=alpha_total_size)
        enc_in.append(eng_one)
        #디코더 입력('S'한글 -> 숫자 -> 원핫인코딩)
        kor =[char_to_num.get(ch) for ch in 'S'+ data[1]]
        kor_one=np.eye(alpha_total_size)[kor]
        dec_in.append(kor_one)
        #디코더 출력(한글'E' -> 숫자)
        kor = [[char_to_num.get(ch)] for ch in data[1]+'E']
        dec_out.append(kor)
        #print(kor)
     #인공신경망에 넣을 데이터이므로 numpy 배열로 전환
    enc_in = np.array(enc_in)
    dec_in = np.array(dec_in)
    dec_out = np.array(dec_out)
    #print(enc_in.shape, dec_in.shape, dec_out.shape)
    return enc_in, dec_in, dec_out

sample=[['cold', '감기'],['come', '오다']]
X_enc,X_dec,Y_dec=encoding(sample)
X_enc.shape,X_dec.shape, Y_dec.shape
    

((2, 4, 171), (2, 3, 171), (2, 3, 1))

# 5. 전체 번역데이터(독립변수,타겟변수)

In [54]:
#Seq2Seq에 들어갈 데이터
X_enc, X_dec, Y_dec = encoding(eng_kor)

In [56]:
X_enc.shape,X_dec.shape, Y_dec.shape

((110, 4, 171), (110, 3, 171), (110, 3, 1))

# 6. 모델구현
- 교안pdf 119p


In [61]:
#인코더 LSTM 구현
ENC_IN = Input(shape=(4,alpha_total_size))
_, state_h,state_c = LSTM(units=MY_HIDDEN, 
                          return_state=True, #디코더 입력을 받기 위한 
                          #return_sequences=False, 
                         )(ENC_IN)
# 인코더와 디코더를 연결할 link
link = [state_h,state_c]
# 디코더 LSTM
DEC_IN = Input(shape=(3,alpha_total_size))
DEC_MID = LSTM(units=MY_HIDDEN, 
             #return_state=False, 기본값  #오른쪽으로 데이터를 넘기지 않음
               return_sequences=True
                )(DEC_IN, initial_state=link)
#최종 출력 층
DEC_OUT= Dense(units=alpha_total_size, activation='softmax')(DEC_MID)
# 모델 
model= Model(inputs=[ENC_IN, DEC_IN],
            outputs=DEC_OUT)
model.summary()


Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_9 (InputLayer)           [(None, 4, 171)]     0           []                               
                                                                                                  
 input_10 (InputLayer)          [(None, 3, 171)]     0           []                               
                                                                                                  
 lstm_8 (LSTM)                  [(None, 128),        153600      ['input_9[0][0]']                
                                 (None, 128),                                                     
                                 (None, 128)]                                                     
                                                                                            

# 7. 모델 학습


In [65]:
model.compile(loss='sparse_categorical_crossentropy',
             optimizer='rmsprop',
             metrics=['accuracy'])
begin= time()
hist = model.fit([X_enc,X_dec],Y_dec,
                epochs=MY_EPOCH,
                verbose=2)
end= time()
print(f'학습시간 : {end-begin:.2f}초')

Epoch 1/500
4/4 - 3s - loss: 6.0833e-07 - accuracy: 1.0000 - 3s/epoch - 858ms/step
Epoch 2/500
4/4 - 0s - loss: 6.0183e-07 - accuracy: 1.0000 - 32ms/epoch - 8ms/step
Epoch 3/500
4/4 - 0s - loss: 5.9424e-07 - accuracy: 1.0000 - 31ms/epoch - 8ms/step
Epoch 4/500
4/4 - 0s - loss: 5.9352e-07 - accuracy: 1.0000 - 30ms/epoch - 8ms/step
Epoch 5/500
4/4 - 0s - loss: 5.9099e-07 - accuracy: 1.0000 - 29ms/epoch - 7ms/step
Epoch 6/500
4/4 - 0s - loss: 5.8810e-07 - accuracy: 1.0000 - 31ms/epoch - 8ms/step
Epoch 7/500
4/4 - 0s - loss: 5.8340e-07 - accuracy: 1.0000 - 30ms/epoch - 8ms/step
Epoch 8/500
4/4 - 0s - loss: 5.8449e-07 - accuracy: 1.0000 - 32ms/epoch - 8ms/step
Epoch 9/500
4/4 - 0s - loss: 5.7798e-07 - accuracy: 1.0000 - 30ms/epoch - 7ms/step
Epoch 10/500
4/4 - 0s - loss: 5.7040e-07 - accuracy: 1.0000 - 32ms/epoch - 8ms/step
Epoch 11/500
4/4 - 0s - loss: 5.6931e-07 - accuracy: 1.0000 - 31ms/epoch - 8ms/step
Epoch 12/500
4/4 - 0s - loss: 5.6498e-07 - accuracy: 1.0000 - 31ms/epoch - 8ms/step
E

Epoch 99/500
4/4 - 0s - loss: 3.7424e-07 - accuracy: 1.0000 - 34ms/epoch - 8ms/step
Epoch 100/500
4/4 - 0s - loss: 3.7172e-07 - accuracy: 1.0000 - 30ms/epoch - 8ms/step
Epoch 101/500
4/4 - 0s - loss: 3.6810e-07 - accuracy: 1.0000 - 33ms/epoch - 8ms/step
Epoch 102/500
4/4 - 0s - loss: 3.6810e-07 - accuracy: 1.0000 - 35ms/epoch - 9ms/step
Epoch 103/500
4/4 - 0s - loss: 3.6810e-07 - accuracy: 1.0000 - 32ms/epoch - 8ms/step
Epoch 104/500
4/4 - 0s - loss: 3.6594e-07 - accuracy: 1.0000 - 23ms/epoch - 6ms/step
Epoch 105/500
4/4 - 0s - loss: 3.6413e-07 - accuracy: 1.0000 - 31ms/epoch - 8ms/step
Epoch 106/500
4/4 - 0s - loss: 3.6630e-07 - accuracy: 1.0000 - 31ms/epoch - 8ms/step
Epoch 107/500
4/4 - 0s - loss: 3.6305e-07 - accuracy: 1.0000 - 30ms/epoch - 8ms/step
Epoch 108/500
4/4 - 0s - loss: 3.6305e-07 - accuracy: 1.0000 - 32ms/epoch - 8ms/step
Epoch 109/500
4/4 - 0s - loss: 3.6449e-07 - accuracy: 1.0000 - 31ms/epoch - 8ms/step
Epoch 110/500
4/4 - 0s - loss: 3.5907e-07 - accuracy: 1.0000 - 22m

Epoch 196/500
4/4 - 0s - loss: 2.7527e-07 - accuracy: 1.0000 - 31ms/epoch - 8ms/step
Epoch 197/500
4/4 - 0s - loss: 2.7635e-07 - accuracy: 1.0000 - 34ms/epoch - 8ms/step
Epoch 198/500
4/4 - 0s - loss: 2.7310e-07 - accuracy: 1.0000 - 31ms/epoch - 8ms/step
Epoch 199/500
4/4 - 0s - loss: 2.7382e-07 - accuracy: 1.0000 - 31ms/epoch - 8ms/step
Epoch 200/500
4/4 - 0s - loss: 2.7165e-07 - accuracy: 1.0000 - 34ms/epoch - 8ms/step
Epoch 201/500
4/4 - 0s - loss: 2.6912e-07 - accuracy: 1.0000 - 32ms/epoch - 8ms/step
Epoch 202/500
4/4 - 0s - loss: 2.7021e-07 - accuracy: 1.0000 - 35ms/epoch - 9ms/step
Epoch 203/500
4/4 - 0s - loss: 2.7093e-07 - accuracy: 1.0000 - 33ms/epoch - 8ms/step
Epoch 204/500
4/4 - 0s - loss: 2.6768e-07 - accuracy: 1.0000 - 35ms/epoch - 9ms/step
Epoch 205/500
4/4 - 0s - loss: 2.6696e-07 - accuracy: 1.0000 - 32ms/epoch - 8ms/step
Epoch 206/500
4/4 - 0s - loss: 2.6949e-07 - accuracy: 1.0000 - 33ms/epoch - 8ms/step
Epoch 207/500
4/4 - 0s - loss: 2.6804e-07 - accuracy: 1.0000 - 41

Epoch 293/500
4/4 - 0s - loss: 2.1458e-07 - accuracy: 1.0000 - 38ms/epoch - 10ms/step
Epoch 294/500
4/4 - 0s - loss: 2.1241e-07 - accuracy: 1.0000 - 30ms/epoch - 8ms/step
Epoch 295/500
4/4 - 0s - loss: 2.1060e-07 - accuracy: 1.0000 - 31ms/epoch - 8ms/step
Epoch 296/500
4/4 - 0s - loss: 2.1133e-07 - accuracy: 1.0000 - 31ms/epoch - 8ms/step
Epoch 297/500
4/4 - 0s - loss: 2.1277e-07 - accuracy: 1.0000 - 30ms/epoch - 8ms/step
Epoch 298/500
4/4 - 0s - loss: 2.0916e-07 - accuracy: 1.0000 - 30ms/epoch - 7ms/step
Epoch 299/500
4/4 - 0s - loss: 2.0880e-07 - accuracy: 1.0000 - 30ms/epoch - 8ms/step
Epoch 300/500
4/4 - 0s - loss: 2.0880e-07 - accuracy: 1.0000 - 30ms/epoch - 8ms/step
Epoch 301/500
4/4 - 0s - loss: 2.0735e-07 - accuracy: 1.0000 - 25ms/epoch - 6ms/step
Epoch 302/500
4/4 - 0s - loss: 2.0844e-07 - accuracy: 1.0000 - 31ms/epoch - 8ms/step
Epoch 303/500
4/4 - 0s - loss: 2.0988e-07 - accuracy: 1.0000 - 31ms/epoch - 8ms/step
Epoch 304/500
4/4 - 0s - loss: 2.0699e-07 - accuracy: 1.0000 - 3

Epoch 390/500
4/4 - 0s - loss: 1.6870e-07 - accuracy: 1.0000 - 33ms/epoch - 8ms/step
Epoch 391/500
4/4 - 0s - loss: 1.7159e-07 - accuracy: 1.0000 - 28ms/epoch - 7ms/step
Epoch 392/500
4/4 - 0s - loss: 1.6870e-07 - accuracy: 1.0000 - 33ms/epoch - 8ms/step
Epoch 393/500
4/4 - 0s - loss: 1.6906e-07 - accuracy: 1.0000 - 31ms/epoch - 8ms/step
Epoch 394/500
4/4 - 0s - loss: 1.6689e-07 - accuracy: 1.0000 - 33ms/epoch - 8ms/step
Epoch 395/500
4/4 - 0s - loss: 1.6725e-07 - accuracy: 1.0000 - 33ms/epoch - 8ms/step
Epoch 396/500
4/4 - 0s - loss: 1.6653e-07 - accuracy: 1.0000 - 32ms/epoch - 8ms/step
Epoch 397/500
4/4 - 0s - loss: 1.6689e-07 - accuracy: 1.0000 - 34ms/epoch - 9ms/step
Epoch 398/500
4/4 - 0s - loss: 1.6689e-07 - accuracy: 1.0000 - 35ms/epoch - 9ms/step
Epoch 399/500
4/4 - 0s - loss: 1.6617e-07 - accuracy: 1.0000 - 35ms/epoch - 9ms/step
Epoch 400/500
4/4 - 0s - loss: 1.6509e-07 - accuracy: 1.0000 - 35ms/epoch - 9ms/step
Epoch 401/500
4/4 - 0s - loss: 1.6364e-07 - accuracy: 1.0000 - 32

Epoch 487/500
4/4 - 0s - loss: 1.4413e-07 - accuracy: 1.0000 - 41ms/epoch - 10ms/step
Epoch 488/500
4/4 - 0s - loss: 1.3908e-07 - accuracy: 1.0000 - 33ms/epoch - 8ms/step
Epoch 489/500
4/4 - 0s - loss: 1.4630e-07 - accuracy: 1.0000 - 39ms/epoch - 10ms/step
Epoch 490/500
4/4 - 0s - loss: 1.4161e-07 - accuracy: 1.0000 - 36ms/epoch - 9ms/step
Epoch 491/500
4/4 - 0s - loss: 1.4269e-07 - accuracy: 1.0000 - 42ms/epoch - 10ms/step
Epoch 492/500
4/4 - 0s - loss: 1.4052e-07 - accuracy: 1.0000 - 32ms/epoch - 8ms/step
Epoch 493/500
4/4 - 0s - loss: 1.3980e-07 - accuracy: 1.0000 - 34ms/epoch - 8ms/step
Epoch 494/500
4/4 - 0s - loss: 1.4088e-07 - accuracy: 1.0000 - 31ms/epoch - 8ms/step
Epoch 495/500
4/4 - 0s - loss: 1.4124e-07 - accuracy: 1.0000 - 30ms/epoch - 7ms/step
Epoch 496/500
4/4 - 0s - loss: 1.4124e-07 - accuracy: 1.0000 - 30ms/epoch - 8ms/step
Epoch 497/500
4/4 - 0s - loss: 1.3944e-07 - accuracy: 1.0000 - 29ms/epoch - 7ms/step
Epoch 498/500
4/4 - 0s - loss: 1.3980e-07 - accuracy: 1.0000 -

In [66]:
model.evaluate([X_enc,X_dec],Y_dec)

4/4 [==============================] - 1s 5ms/step - loss: 1.3836e-07 - accuracy: 1.0000


[1.3835501988523902e-07, 1.0]

# 8. 모델 사용

In [69]:
# 쉬운문제
easy_test=[['very','pp'],
           ['wave','pp'],
           ['wood','pp'],
           ['sign','pp'],
           ['thin','pp']]
enc_in, dec_in, _ = encoding(easy_test)
enc_in.shape, dec_in.shape, _.shape

((5, 4, 171), (5, 3, 171), (5, 3, 1))

In [72]:
pred = model.predict([enc_in,dec_in])
y_hat=pred.argmax(axis=-1)

1/1 [==============================] - 0s 29ms/step


array([[ 75, 124,   1],
       [157,  62,   1],
       [ 48,  83,   1],
       [110, 135,   1],
       [113, 129,   1]], dtype=int64)

In [80]:
for i in range(len(easy_test)):
    eng = easy_test[i][0]
    hat=pred[i].argmax(axis=-1)
    kor = ''.join([alpha[num] for num in hat[:-1]])
    print(f'{eng} => {kor}{hat[:-1]}')

very => 매우[ 75 124]
wave => 파도[157  62]
wood => 나무[48 83]
sign => 싸인[110 135]
thin => 얇은[113 129]


In [81]:
# 어려운 문제
easy_test=[['vrey','pp'],
           ['waev','pp'],
           ['wodd','pp'],
           ['sigg','pp'],
           ['tine','pp']]
enc_in, dec_in, _ = encoding(easy_test)
enc_in.shape, dec_in.shape, _.shape

((5, 4, 171), (5, 3, 171), (5, 3, 1))

In [82]:
pred = model.predict([enc_in,dec_in])
y_hat=pred.argmax(axis=-1)

1/1 [==============================] - 0s 30ms/step


In [83]:
for i in range(len(easy_test)):
    eng = easy_test[i][0]
    hat=pred[i].argmax(axis=-1)
    kor = ''.join([alpha[num] for num in hat[:-1]])
    print(f'{eng} => {kor}{hat[:-1]}')

vrey => 매우[ 75 124]
waev => 파도[157  62]
wodd => 나무[48 83]
sigg => 싸인[110 135]
tine => 작은[139 129]
